# `metrics.py`

## Function summary


```text
recall_sims:            (N, n_sims) recall matrix
N:                      list length
boundary_positions:     1-based boundary indices
│
│
├── Whole-list metrics
│   │
│   ├── compute_spc(recall_sims, N)
│   │           └── P(recall) at each serial position
│   │
│   ├── compute_pfr(recall_sims, N)
│   │           └── P(first recall) at each serial position
│   │
│   ├── compute_lag_crp(recall_sims, N)
│   │           └── whole-list opportunity-corrected lag-CRP
│   │
│   └── recall_accuracy(recall_sims, N, unique=True)
│               └── mean fraction of unique items recalled
│
├── Train-level metrics (Polyn 2009)
│   │
│   ├── _train_bounds_from_boundaries(N, boundary_positions)
│   │           └── derive (start, end) tuples for each train
│   │
│   ├── compute_train_recall(recall_sims, N, boundary_positions, unique=True)
│   │           └── proportion recalled per train, per simulation
│   │               returns train_ids, bounds, mean, SE, per-sim matrix
│   │
│   └── mean_train_recall(recall_sims, N, boundary_positions, unique=True)
│               └── unweighted scalar mean across trains
│                   (equals recall_accuracy only when trains are equal length)
│
├── Position-specific metrics
│   │       
│   ├── SPC around one chosen center
│   │      │
│   │      └── local_spc(recall_sims, N, center, half_window)
│   │               │
│   │               ├── call once: compute_spc(recall_sims, N)
│   │               │
│   │               └── slice SPC in a local window around one center
│   │
│   └── CRP around one chosen center
│          └── position_conditional_crp(recall_sims, N, start, lag)
│   
└── Pool metrics across boundaries
    │
    ├── CRP around all boundaries
    │       └── boundary_transition(recall_sims,N,start_offset,target_offset,boundary_positions)
    │               │
    │               ├── for each boundary j:
    │               │       start  = j + start_offset
    │               │       target = j + target_offset
    │               │       lag    = target - start
    │               │
    │               ├── call: position_conditional_crp(recall_sims, N, start, lag)
    │               │
    │               └── pool numerators / denominators across all boundaries
    │
    │
    └── SPC around all boundaries
            │
            └── boundary_local_spc(recall_sims, N, boundary_positions, half_window)
                    │
                    ├── call once: compute_spc(recall_sims, N)
                    │
                    ├── for each boundary j:
                    │       convert relative positions to absolute positions
                    │       extract local SPC window from full SPC
                    │
                    └── average local SPC curves across boundaries
```

```text
Summary panel
    └── summarize_condition_metrics(recall_sims, N, boundary_positions, half_window)
            │
            ├── call whole-list metrics:
            │       compute_spc
            │       compute_pfr
            │       compute_lag_crp
            │       recall_accuracy
            │
            ├── call train-level metrics:
            │       compute_train_recall
            │       mean_train_recall (derived from compute_train_recall output)
            │
            ├── call position-specific metrics:
            │        │
            │        ├── CRP around all boundaries
            │        │        └── boundary_transition(..., -1,  0)   # (j-1) → j
            │        │            boundary_transition(...,  0, -1)   # j → (j-1)
            │        │            boundary_transition(...,  0,  1)   # j → (j+1)
            │        │
            │        └── SPC around all boundaries
            │                 └── boundary_local_spc(...)
            │
            └── return one dictionary of all outputs
                    includes: train_recall, train_bounds, mean_train_recall
```

# metrics.py pseudocodes

## `compute_spc`

### Pseudocode
- Initialize an array of recall probabilities, one slot per serial position.
- For each serial position `j`, check across simulations whether `j` appeared anywhere in the recall sequence.
- Store the mean of that boolean indicator as the SPC value for `j`.
- Return the full SPC vector.

| Pseudocode step | Code |
|---|---|
| Initialize output vector | `spc = np.zeros(N)` |
| Loop over serial positions 1…N | `for j in range(1, N + 1):` |
| Compute mean recall presence for each position | `spc[j - 1] = np.mean(np.any(recall_sims == j, axis=0))` |
| Return SPC | `return spc` |

## `compute_pfr`

### Pseudocode
- Take the first recalled item from each simulation.
- Drop zero entries so only valid first recalls remain.
- Initialize a probability vector over serial positions.
- For each serial position, compute the proportion of trials where that item was recalled first.
- Return the PFR vector.

| Pseudocode step | Code |
|---|---|
| Extract first recall per trial | `first = recall_sims[0, :]` |
| Remove zero / empty recalls | `first = first[first > 0]` |
| Initialize PFR vector | `pfr = np.zeros(N)` |
| Guard against empty first-recall set | `if len(first) > 0:` |
| Loop over serial positions | `for j in range(1, N + 1):` |
| Compute probability of first recall | `pfr[j - 1] = np.mean(first == j)` |
| Return PFR | `return pfr` |

## `compute_lag_crp`

### Pseudocode
- Build the full lag axis from `-(N-1)` to `+(N-1)`.
- Initialize numerator and denominator counts for each lag.
- For each simulation, clean the recall sequence by removing zero entries.
- Walk through consecutive recall transitions.
- At each transition, define the set of still-available items and add one opportunity count to every lag those items would produce from the current item.
- Add one observed count to the actual lag taken by the next recall.
- Convert counts to probabilities wherever the denominator is positive.
- Return the lag axis and CRP values.

| Pseudocode step | Code |
|---|---|
| Create lag axis | `max_lag = N - 1; lag_vals = np.arange(-max_lag, max_lag + 1)` |
| Initialize numerator / denominator | `numer = np.zeros(…)` / `denom = np.zeros(…)` |
| Map lag to array index | `lag_to_idx = {L: i for i, L in enumerate(lag_vals)}` |
| Loop over simulations; clean sequence | `for s in …: seq = …; seq = seq[seq > 0]…` |
| Track recalled items | `recalled = set()` |
| Loop over transitions; tally opportunities and observed lag | `for t in …: recalled.add(cur); remaining = …; denom[…] += 1; numer[…] += 1` |
| Compute CRP only where opportunities exist | `crp = np.zeros_like(numer); valid = denom > 0; crp[valid] = numer[valid] / denom[valid]` |
| Return lag axis and CRP | `return lag_vals, crp` |

## `recall_accuracy`

### Pseudocode
- Return `NaN` if the recall matrix is missing or empty.
- For each simulation, remove zero entries from the recall sequence.
- If nothing was recalled, record accuracy as zero.
- Optionally collapse repeated recalls to unique recalled items.
- Compute recalled-items fraction as `len(recalled_items) / N`.
- Average that fraction across simulations.

| Pseudocode step | Code |
|---|---|
| Guard against empty input | `if recall_sims is None or recall_sims.size == 0: return np.nan` |
| Initialize per-trial accuracy list | `acc = []` |
| Loop over simulations | `for s in range(recall_sims.shape[1]):` |
| Remove zero entries | `seq = recall_sims[:, s]; seq = seq[seq > 0].astype(int)` |
| Handle empty-recall trial | `if seq.size == 0: acc.append(0.0); continue` |
| Optionally unique the recalled items | `if unique: seq = np.unique(seq)` |
| Compute per-trial fraction recalled | `acc.append(len(seq) / float(N))` |
| Return mean accuracy | `return float(np.mean(acc)) if acc else np.nan` |

## `_train_bounds_from_boundaries`

### Pseudocode
- Build a list of train start positions: position 1 plus each boundary position.
- For each start, compute the end as the position before the next start (or `N` for the last train).
- Skip any train where start > end (empty-train edge case).
- Return the list of `(start, end)` tuples in 1-based inclusive indexing.

| Pseudocode step | Code |
|---|---|
| Build ordered list of train starts | `starts = [1] + sorted(boundary_positions)` |
| Loop over starts and compute ends | `for i in range(len(starts)): start = starts[i]; end = starts[i+1] - 1 if i+1 < len(starts) else N` |
| Skip empty trains | `if start <= end: bounds.append((start, end))` |
| Return bounds | `return bounds` |

### Example

For `N = 32` and `boundary_positions = [9, 17, 25]`:

| Train | Start | End | Length |
|---|---|---|---|
| 1 | 1 | 8 | 8 |
| 2 | 9 | 16 | 8 |
| 3 | 17 | 24 | 8 |
| 4 | 25 | 32 | 8 |

## `compute_train_recall`

### Pseudocode
- Derive train bounds from the boundary positions.
- For each simulation, remove zero-padding from the recall sequence and optionally unique the recalled items.
- For each train, count how many of the recalled items fall within that train's position range.
- Divide by train length to get the proportion recalled.
- After processing all simulations, compute the mean and standard error across simulations for each train.
- Return train IDs, bounds, per-train means, per-train SEs, and the full per-simulation matrix.

| Pseudocode step | Code |
|---|---|
| Derive train bounds | `bounds = _train_bounds_from_boundaries(N, boundary_positions)` |
| Allocate per-sim × per-train matrix | `per_sim_train_recall = np.zeros((n_sims, n_trains))` |
| Loop over simulations | `for s in range(n_sims):` |
| Remove zero-padding, optionally unique | `recalls = recalls[recalls > 0]; if unique: recalls = np.unique(recalls)` |
| Count hits within each train's range | `hits = np.sum((recalls >= start) & (recalls <= end))` |
| Compute proportion | `per_sim_train_recall[s, t] = hits / train_length` |
| Compute mean and SE across simulations | `mean_tr = np.mean(…, axis=0); se_tr = np.std(…, ddof=1) / np.sqrt(n_sims)` |
| Return full results | `return train_ids, bounds, mean_tr, se_tr, per_sim_train_recall` |

### Interpretive caveats

- **Train 1 is special:** it starts at position 1, so its recall may partly reflect primacy dynamics rather than pure boundary structure. Differences among Trains 2–4 are more directly informative about the manipulated boundary/baseline drift schedule.
- **Zero-padding is excluded:** `recalls > 0` ensures that the `0` values used to pad unused recall slots are never counted as recalled items.
- **Unique by default:** repeated recalls of the same item within a simulation are collapsed before counting, matching the convention used by `recall_accuracy`.

## `mean_train_recall`

### Pseudocode
- Call `compute_train_recall` to get the per-train mean recall proportions.
- Return the unweighted average across trains as a scalar.

| Pseudocode step | Code |
|---|---|
| Compute per-train means | `_, _, mean_tr, _, _ = compute_train_recall(…)` |
| Average across trains | `return float(np.mean(mean_tr))` |

### Relationship to `recall_accuracy`

`mean_train_recall` equals `recall_accuracy` **only when all trains have the same length**. For the default 32-item design with boundaries at [9, 17, 25], all four trains have length 8, so the two values coincide. For configurations with unequal trains (e.g., a single boundary at position 13 producing a 12-item and a 20-item train), they will differ because `mean_train_recall` weights each train equally regardless of length.

## `position_conditional_crp`

### Pseudocode
- Set target = `start_pos + target_lag`.
- Skip if target is outside list bounds.
- For each trial:
  - remove zero-padding from the recall sequence
  - scan recall transitions one step at a time
- Whenever current recall == `start_pos`
  - and the target has not yet been recalled:
    - add 1 to the denominator
- If the next recall == target:
  - add 1 to the numerator
- Return the opportunity-corrected transition probability
  - or return raw counts as well, if requested.

| Pseudocode step | Code |
|---|---|
| Define target from start position and lag | `target_pos = start_pos + target_lag` |
| Exit early if target is outside list bounds | `if target_pos < 1 or target_pos > N: … return np.nan` |
| Initialize hit and opportunity counts | `num = 0; den = 0` |
| Loop over trials and clean each recall sequence | `for s in …: seq = …; if len(seq) < 2: continue` |
| Step through recall transitions while tracking already recalled items | `recalled = set(); for t in …: cur = seq[t]; nxt = seq[t+1]; recalled.add(cur)` |
| Count an opportunity when `cur == start_pos` and the target is still available | `if cur == start_pos: if target_pos not in recalled: den += 1` |
| Count a hit when the next recalled item is exactly the target | `if nxt == target_pos: num += 1` |
| Convert counts to probability | `prob = (num / den) if den > 0 else np.nan` |
| Return either probability only or probability with counts | `if return_counts: return TransitionResult(prob, num, den); return prob` |

## `boundary_transition`

### Pseudocode
- Initialize pooled numerator and denominator.
- For each boundary position `j`, compute the absolute start and target positions from the given offsets.
- Skip any pair where either position falls outside the list.
- Compute the lag between start and target, then call the low-level opportunity-corrected CRP engine to get raw counts.
- Pool those counts across all boundaries.
- Return the pooled probability, optionally with counts.

| Pseudocode step | Code |
|---|---|
| Initialize pooled counts | `total_num = 0; total_den = 0` |
| Loop over boundary positions | `for j in boundary_positions:` |
| Compute absolute start and target | `start = j + start_offset; target = j + target_offset` |
| Skip out-of-range positions | `if start < 1 or start > N: continue; if target < 1 or target > N: continue` |
| Compute lag and call low-level CRP | `lag = target - start; tr = position_conditional_crp(…, return_counts=True)` |
| Accumulate raw counts | `total_num += tr.num; total_den += tr.den` |
| Compute pooled probability | `prob = (total_num / total_den) if total_den > 0 else np.nan` |
| Return TransitionResult or probability | `if return_counts: return TransitionResult(…); return prob` |

## `local_spc`

### Pseudocode
- Define a local window around a chosen center position.
- Clip the window so it stays inside the list.
- Compute the full-list SPC.
- Return only the positions and SPC values inside the chosen local window.

| Pseudocode step | Code |
|---|---|
| Compute lower / upper bounds of window | `lo = max(1, center - half_window); hi = min(N, center + half_window)` |
| Build 1-based position range | `pos_range = np.arange(lo, hi + 1)` |
| Compute full SPC | `spc_full = compute_spc(recall_sims, N)` |
| Return local positions and local SPC slice | `return pos_range, spc_full[pos_range - 1]` |

## `boundary_local_spc`

### Pseudocode
- Compute whole-list SPC once.
- Create a relative-position axis centered on the boundary.
- For each boundary, convert those relative positions into absolute serial positions.
- Mask positions that would fall outside the list and fill them with `NaN`.
- Extract the SPC values for valid positions to form one local SPC curve per boundary.
- Average local SPC curves across boundaries and compute the standard error across boundaries.
- Return relative positions, mean local SPC, and standard error.

| Pseudocode step | Code |
|---|---|
| Compute whole-list SPC | `spc_full = compute_spc(recall_sims, N)` |
| Create relative-position axis | `rel = np.arange(-half_window, half_window + 1)` |
| Loop over boundaries; extract local curves | `for j in …: abs_pos = j + rel; valid = …; curve[valid] = spc_full[…]; curves.append(curve)` |
| Stack curves into array | `curves = np.array(curves)` |
| Compute mean across boundaries | `mean_spc = np.nanmean(curves, axis=0)` |
| Compute standard error across boundaries | `se_spc = np.nanstd(…) / np.sqrt(np.sum(~np.isnan(…), axis=0))` |
| Return relative positions, mean, SE | `return rel, mean_spc, se_spc` |

## `summarize_condition_metrics`

### Pseudocode
- Compute whole-list lag-CRP.
- Compute boundary-local SPC summary.
- Compute train-level recall (per-train means, SEs, and bounds).
- Derive the scalar mean-train-recall convenience value.
- Bundle whole-list metrics, train-level metrics, pooled boundary transition metrics, and local SPC into one dictionary.
- Return that dictionary as a compact summary for one condition.

| Pseudocode step | Code |
|---|---|
| Compute whole-list lag-CRP | `lag_vals, crp = compute_lag_crp(recall_sims, N)` |
| Compute boundary-local SPC | `rel, mean_spc_local, se_spc_local = boundary_local_spc(…)` |
| Compute train-level recall | `train_ids, bounds, mean_tr, se_tr, _ = compute_train_recall(…)` |
| Derive scalar mean-train-recall | `mtr = float(np.mean(mean_tr))` |
| Build summary dictionary | `return {"recall_accuracy": …, "whole_spc": …, …, "train_recall": …, "train_bounds": …, "mean_train_recall": …}` |

### Output dictionary keys

| Key | Value |
|---|---|
| `recall_accuracy` | scalar float |
| `whole_spc` | `(N,)` float array |
| `whole_pfr` | `(N,)` float array |
| `whole_lag_crp` | `(lag_vals, crp)` tuple |
| `pre_to_boundary` | scalar float |
| `boundary_backward` | scalar float |
| `boundary_forward` | scalar float |
| `boundary_local_spc` | `(rel, mean_spc, se_spc)` tuple |
| `train_recall` | `(train_ids, mean_tr, se_tr)` tuple |
| `train_bounds` | list of `(start, end)` tuples |
| `mean_train_recall` | scalar float |

# `simulation.py`

## Flowchart

```text
Choose hypothesis / condition
│
├── H1: boundary-update manipulation
│   └── build_boundary_schedule(B_non, delta, boundary_positions)
│           │
│           └── returns B_encD
│
├── H2: global tonic manipulation
│   └── build_global_schedule(B_non, B_boundary, boundary_positions)
│           │
│           └── returns B_encD
│
└── Baseline condition
    └── build_baseline_schedule(boundary_positions)
            │
            └── internally calls:
                build_boundary_schedule(...)
                    │
                    └── returns B_encD
```

```text
B_encD = (N,) float drift vector
│
├── simulate_single_trial(B_encD, rng, ...)
│       └── one encode→retrieve trial
│           returns:
│           - recalls
│           - times
│           - net_w_fc
│           - net_w_cf
│
└── run_batch(B_encD, n_sims, seed, ...)
        └── repeatedly calls simulate_single_trial(...)
            stacks results across trials
            returns:
            - label
            - B_encD
            - recall_sims
            - times_sims
            - net_w_fc
            - net_w_cf
```

## Function summary

| Function | Purpose | Key inputs | Input structure | Output | Output structure |
|---|---|---|---|---|---|
| `build_boundary_schedule` | Construct an H1 drift schedule with fixed non-boundary drift and additive boundary boost | `B_non`, `delta`, `boundary_positions` | `B_non`: float baseline drift for non-boundary items; `delta`: float additive boost at boundaries; `boundary_positions`: list or array of 1-based boundary positions | Boundary-boosted drift schedule | `(N,)` float array of per-position drift values, clipped to `[0, 1]` |
| `build_global_schedule` | Construct an H2 drift schedule with globally shifted non-boundary drift and fixed boundary drift | `B_non`, `B_boundary`, `boundary_positions` | `B_non`: float drift for non-boundary items; `B_boundary`: float drift for boundary items; `boundary_positions`: list or array of 1-based boundary positions | Global-tonic drift schedule | `(N,)` float array of per-position drift values, clipped to `[0, 1]` |
| `build_baseline_schedule` | Construct the default healthy baseline drift schedule from config defaults | `boundary_positions` | `boundary_positions`: list or array of 1-based boundary positions | Baseline drift schedule | `(N,)` float array of per-position drift values |
| `describe_schedule` | Give a short human-readable summary of a drift schedule | `B_encD`, `boundary_positions` | `B_encD`: `(N,)` float array of drift values; `boundary_positions`: list or array of 1-based boundary positions | Schedule description | Multi-line string summarizing list length, boundary locations, mean/range of non-boundary drift, and mean/range of boundary drift |
| `_dot` | Compute scalar dot product of two vectors / column arrays | `a`, `b` | Array-like vectors or column arrays with compatible shapes | Scalar dot product | `float` |
| `_norm` | Compute Euclidean norm of a vector | `v` | Array-like vector | Vector norm | `float` |
| `simulate_single_trial` | Run one full CMR encode→retrieve trial under a specified drift schedule | `B_encD`, `rng`, `gamma_fc`, `eta`, `B_rec` | `B_encD`: `(N,)` float array of per-position encoding drift rates; `rng`: NumPy random generator; optional overrides: `gamma_fc`, `eta`, `B_rec` as floats or `None` | One trial’s recall outputs and final matrices | Tuple: `recalls = (N,)` int array of 1-based recalled serial positions with `0` padding; `times = (N,)` float array of cumulative recall times; `net_w_fc = (N, N)` float matrix; `net_w_cf = (N, N)` float matrix |
| `run_batch` | Run many CMR trials under one fixed drift schedule and collect batch outputs | `B_encD`, `n_sims`, `seed`, `label`, `sim_kwargs` | `B_encD`: `(N,)` float array; `n_sims`: int number of trials; `seed`: int RNG seed; `label`: string condition name; `sim_kwargs`: optional parameter overrides forwarded to `simulate_single_trial` | Batch simulation bundle | `dict` with keys: `label` (str), `B_encD` (`(N,)` float array), `recall_sims` (`(N, n_sims)` int array), `times_sims` (`(N, n_sims)` float array), `net_w_fc` (`(N, N)` float matrix), `net_w_cf` (`(N, N)` float matrix) |

# simulation.py pseudocodes

## `build_boundary_schedule`

### Pseudocode
- Create a length-`N` vector filled with the non-boundary baseline drift.
- For every boundary position, replace the corresponding entry with `B_non + delta`.
- Clip the resulting schedule to `[0, 1]` and return it.

| Pseudocode step | Code |
|---|---|
| Initialize all positions to baseline drift | `B = np.full(N, B_non, dtype=float)` |
| Overwrite boundary positions with boosted drift | `for j in boundary_positions: B[j - 1] = B_non + delta` |
| Clip and return schedule | `return np.clip(B, 0.0, 1.0)` |

## `build_global_schedule`

### Pseudocode
- Create a length-`N` vector filled with the chosen non-boundary drift.
- For every boundary position, replace the entry with the chosen boundary drift.
- Clip to `[0, 1]` and return.

| Pseudocode step | Code |
|---|---|
| Initialize all positions to non-boundary drift | `B = np.full(N, B_non, dtype=float)` |
| Overwrite all boundary positions | `for j in boundary_positions: B[j - 1] = B_boundary` |
| Clip and return schedule | `return np.clip(B, 0.0, 1.0)` |

## `build_baseline_schedule`

The **baseline** is the default boundary schedule:
- default non-boundary drift
- default non-boundary drift + default boundary boost
> Equivalent to calling `build_boundary_schedule(...)` with the default config values.


| General boundary schedule | non-boundary = `B_non`; boundary = `B_non + delta` |
|---|---|
| Baseline schedule | non-boundary = `B_NON_BOUNDARY_BASE`; boundary = `B_NON_BOUNDARY_BASE + B_BOUNDARY_DELTA_BASE` |

## `describe_schedule`

### Pseudocode
- Convert the boundary list into a set for membership checks.
- Split the drift vector into non-boundary values and boundary values.
- Compute summary statistics for each group.
- Format those statistics into a short human-readable multi-line string.

| Pseudocode step | Code |
|---|---|
| Create set of boundary positions | `bps = set(boundary_positions)` |
| Collect non-boundary drift values | `non_vals = [B_encD[i] for i in range(len(B_encD)) if (i + 1) not in bps]` |
| Collect boundary drift values | `bdy_vals = [B_encD[j - 1] for j in boundary_positions]` |
| Assemble summary strings | `parts = [f"N = {len(B_encD)}, …", f"non-boundary drift: …", f"boundary drift: …"]` |
| Join and return description | `return "\n".join(parts)` |

### Why both `bps` and `boundary_positions` are used

- `bps = set(boundary_positions)`
  - used for **filtering**
  - supports fast membership checks when scanning all positions:
    - keep positions where `(i + 1) not in bps`

- `boundary_positions`
  - used for **direct extraction**
  - no membership test is needed, because these are already the known boundary positions:
    - loop through each `j` and pull out `B_encD[j - 1]`

| Object | Role |
|---|---|
| `bps = set(boundary_positions)` | filter the full drift vector into **non-boundary** values |
| `boundary_positions` | directly extract the **boundary** drift values |

## `_dot`

### Pseudocode
- Compute the scalar dot product of two vectors / column arrays and return it as a float.

| Pseudocode step | Code |
|---|---|
| Dot product and scalar conversion | `return float((np.asarray(a).T @ np.asarray(b)).ravel()[0])` |

## `_norm`

### Pseudocode
- Compute the Euclidean norm of a vector and return it as a float.

| Pseudocode step | Code |
|---|---|
| Norm and scalar conversion | `return float(np.linalg.norm(v))` |

## `simulate_single_trial`

### Pseudocode
- Read default parameters from configuration and optionally override selected ones.
- Precompute encoding and retrieval constants, and cast the drift schedule to a float array.
- Initialize item features, context state, and the two associative weight matrices.
- Encoding phase: for each study position, activate the studied item feature, compute incoming context, update context using the drift value for that position, and write new associations into `M_FC` and `M_CF`.
- Initialize recall outputs, retrieved-item mask, thresholds, and the mixed episodic/semantic retrieval weights.
- Retrieval phase: repeatedly accumulate evidence over short cycles using competition and noise until an item crosses threshold or recall time expires.
- When an item wins, map the feature index back to serial position, update context using recall drift, update the retrieval matrices if enabled, record the recalled serial position and time, and mark the item as retrieved.
- Continue until recall time runs out, then return recalled positions, times, and final weight matrices.

| Pseudocode step | Code |
|---|---|
| Read / override parameters | `p = BASE_PARAMS; gamma_fc = … if … else p["gamma_fc"]; …; B_encD = np.asarray(B_encD, dtype=float)` |
| Initialize context, features, and matrices | `net_f = np.zeros((N,1)); net_c = np.zeros((N,1)); net_w_fc = np.eye(N)*eye_fc; net_w_cf = np.zeros((N,N))` |
| Encoding loop | `for pos in range(N): … net_c = rho*net_c + B*net_c_in; net_w_fc += …; net_w_cf += …` |
| Initialize retrieval state | `recalls = np.zeros(N, dtype=int); … net_weights = episodic_w*net_w_cf + sem_w*sem_mat; …` |
| Outer retrieval loop over remaining recall time | `while time_passed < rec_time: f_in = net_weights @ net_c; … time_passed += i*dt` |
| Threshold crossing / winner selection | `while i < max_cycles and not crossed: x = x + …; … if np.any(x[retrievable] >= thresholds[…]): crossed = True; …` |
| Process winning recall and update state | `if crossed and winners is not None: winner = …; net_c = rho*net_c + B_rec*net_c_in; recalls[…] = sp1; …` |
| Return recalls, times, and final matrices | `return recalls, times, net_w_fc, net_w_cf` |

## `run_batch`

### Pseudocode
- Initialize a random-number generator and cast the drift schedule to an array.
- Allocate arrays to store recalled positions and recall times across simulations.
- Loop over simulations and run one trial each time using the same drift schedule.
- Store recall outputs and keep the final weight matrices from the last run.
- Return all stored results in a dictionary.

| Pseudocode step | Code |
|---|---|
| Initialize RNG and drift schedule array | `rng = np.random.default_rng(seed); B = np.asarray(B_encD, dtype=float)` |
| Allocate batch output arrays | `recall_sims = np.zeros((N, n_sims), dtype=int); times_sims = …; wfc_last = None; wcf_last = None` |
| Loop over simulations | `for s in range(n_sims): rec, t, wfc, wcf = simulate_single_trial(B, rng, **sim_kwargs); …` |
| Return results dictionary | `return {"label": label, "B_encD": B, "recall_sims": …, "times_sims": …, …}` |